In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

news_df = pd.read_csv('/content/drive/MyDrive/MRP/news_preprocessed.csv', parse_dates=['date'])
news_df = news_df.sort_values(['symbol', 'date']).reset_index(drop=True)
news_df.head(), news_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1848357 entries, 0 to 1848356
Data columns (total 5 columns):
 #   Column               Dtype         
---  ------               -----         
 0   date                 datetime64[ns]
 1   symbol               object        
 2   avg_sentiment        float64       
 3   avg_sentiment_score  float64       
 4   news_count           int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 70.5+ MB


(        date symbol  avg_sentiment  avg_sentiment_score  news_count
 0 2016-01-06      A            0.0             0.999295           2
 1 2016-01-07      A            0.5             0.992147           2
 2 2016-02-05      A            0.0             0.997771           1
 3 2016-02-10      A            0.0             0.999134           1
 4 2016-02-15      A            0.0             0.999934           1,
 None)

In [3]:
# had_news & news_count
# Binary flag: 1 if any articles exist, else 0
news_df['had_news'] = (news_df['news_count'] > 0).astype(int)


In [4]:
# Day-over-day change in article volume per symbol
news_df['delta_news_count'] = news_df.groupby('symbol')['news_count'] \
                                     .diff().fillna(0)


In [5]:
print(news_df[['symbol','date','news_count','had_news','delta_news_count']].head(10))

  symbol       date  news_count  had_news  delta_news_count
0      A 2016-01-06           2         1               0.0
1      A 2016-01-07           2         1               0.0
2      A 2016-02-05           1         1              -1.0
3      A 2016-02-10           1         1               0.0
4      A 2016-02-15           1         1               0.0
5      A 2016-02-16           7         1               6.0
6      A 2016-02-17           3         1              -4.0
7      A 2016-02-23           1         1              -2.0
8      A 2016-02-24           1         1               0.0
9      A 2016-02-29           1         1               0.0


In [6]:
# Align Weekend News to Next Business Day & Day-of-Week
import pandas as pd
from pandas.tseries.offsets import BDay

# Shift any weekend dates (Sat=5, Sun=6) to the following Monday
weekend_mask = news_df['date'].dt.dayofweek >= 5
news_df.loc[weekend_mask, 'date'] = news_df.loc[weekend_mask, 'date'] + BDay(1)

# Now extract day-of-week (0–4 only, since we've shifted weekends)
news_df['day_of_week'] = news_df['date'].dt.dayofweek

# Preview to confirm no 5/6 values remain
print(news_df['day_of_week'].value_counts().sort_index())

day_of_week
0    470469
1    351202
2    354337
3    357973
4    314376
Name: count, dtype: int64


In [8]:
print(news_df[['date','day_of_week']].head(10))


        date  day_of_week
0 2016-01-06            2
1 2016-01-07            3
2 2016-02-05            4
3 2016-02-10            2
4 2016-02-15            0
5 2016-02-16            1
6 2016-02-17            2
7 2016-02-23            1
8 2016-02-24            2
9 2016-02-29            0


In [9]:
# Save Dataset
news_df.to_csv('/content/drive/MyDrive/MRP/news_features_engineered.csv', index=False)
print("Saved news_with_features.csv with columns:")
print(news_df.columns.tolist())

Saved news_with_features.csv with columns:
['date', 'symbol', 'avg_sentiment', 'avg_sentiment_score', 'news_count', 'had_news', 'delta_news_count', 'day_of_week']
